# Python System Interpreter

This notebook introduces a common Python packaging problem on a shared Linux machine. Bob and Alice both work on the same computer and both rely on the same system interpreter. Bob is the administrator for this shared machine, so he can install or replace packages in the system Python environment. Alice is a normal user, so she cannot modify system packages directly. This difference matters because the Python interpreter is shared, but the users and their home directories are separate.

| User    | Home           | Sudo |
| ------- | -------------- | ---- |
| `bob`   | `/home/bob`    | yes  |
| `alice` | `/home/alice`  | no   |

This arrangement creates dependency conflicts quickly. A package change made for one user can break the other user's project, and Alice's own work becomes hard to manage when her multiple projects need different versions of the same dependency. One shared Python environment cannot safely satisfy all of those conflicting requirements at the same time.

This notebook focuses on these **dependency-conflict scenarios**:

- **Shared machine conflict.** Two users share one system Python installation.
- **Cross-project conflict.** Alice's own projects can still fight over incompatible versions when they share one user-level environment.
- **Isolation progression.** We then compare system packages, `pip install --user`, and project-specific virtual environments.

---

## Inspect the Environment

### Bash Helpers

The first Bash block writes reusable Bash helpers to a shell script.

In [ ]:
%%bash
set -euo pipefail
cat > /tmp/bash_helpers.sh <<'EOF'
show_user() {
  local username=$1
  echo "$username"
  printf '%*s\n' "${#username}" '' | tr ' ' '-'
  getent passwd "$username"
  groups "$username"
  sudo su - "$username" -c '
whoami
id
echo "HOME=$HOME"
python3 -c "import sys; print(sys.executable)"
'
}

show_subfolders() {
  local directory=$1
  echo "$directory"
  shopt -s nullglob
  for path in "$directory"/*; do
    [[ -d "$path" ]] || continue
    printf '  - %s\n' "$(basename "$path")"
  done
}
EOF
cat /tmp/bash_helpers.sh

### Inspect Registered Users

Load the above helper file to inspect users in a consistent format.

In [ ]:
%%bash
set -euo pipefail
source /tmp/bash_helpers.sh

# Show Bob user
echo ""
show_user bob
sudo su - bob -c 'sudo -n id'

# Alice User
echo ""
show_user alice
sudo su - alice -c 'sudo -n id' || true


### Explore System Interpreter

This Bash block inspects the **System Python Environment** of the underlying shared machine.

In [ ]:
%%bash
set -euo pipefail
source /tmp/bash_helpers.sh
LOCAL_SITE=$(python3 -c "import sysconfig; print(sysconfig.get_path('purelib'))")

# Show Python version
echo
echo 'System python version'
echo '---------------------'
python3 --version

# Show the APT-controlled packages
echo 'APT-controlled Python package directories'
echo '---------------------------------------'
show_subfolders /usr/lib/python3/dist-packages
echo

# Show the system packages
echo 'Administrator-installed Python package directories'
echo '------------------------------------------------'
show_subfolders "$LOCAL_SITE"
echo
echo 'Installed packages from the system interpreter'
echo '---------------------------------------------'
python3 -m pip list
echo

# Switch to Alice and inspect her user packages
echo 'Alice user-installed pip packages'
echo '--------------------------------'
sudo su - alice -c '
python3 -m site --user-site
python3 -m pip list --user
'


---

## Setting Up the Demo Projects

This shared machine hosts **Bob's legacy server**, **Alice FastAPI v1**, and **Alice FastAPI v2**.

These three projects expose two dependency conflicts. Bob's legacy server depends on `requests==2.0.0`, while Alice's projects need `requests==2.32.3`. Alice's own FastAPI projects also disagree with each other because one needs the older FastAPI and Pydantic v1 stack, while the other needs the newer FastAPI and Pydantic v2 stack.

In [ ]:
from pathlib import Path

### Bob's legacy server

This Python cell keeps Bob's application embedded in the notebook and writes a temporary source file that the next Bash cell will place onto the shared machine.

In [ ]:
BOB_LEGACY = Path('/home/bob/projects/legacy-server')
BOB_TMP = Path('/tmp/legacy-server-main.py')

bob_code = '''# Bob's legacy server.
import sys
import requests

print('Bob Legacy Server')
print(f'  Python  : {sys.executable}')
print(f'  requests: {requests.__version__}')

req = requests.Request('GET', 'https://example.invalid/')
print(f'  built a  {req.method} request for {req.url}')
'''

BOB_TMP.write_text(bob_code, encoding='utf-8')

With Bob's source string prepared, this Bash block creates the project directory on the host and copies `main.py` into place.

In [ ]:
%%bash
set -euo pipefail
sudo mkdir -p /home/bob/projects/legacy-server
sudo cp /tmp/legacy-server-main.py /home/bob/projects/legacy-server/main.py
sudo chown -R bob:bob /home/bob/projects
ls -la /home/bob/projects/legacy-server
echo
echo 'main.py'
echo '-------'
cat /home/bob/projects/legacy-server/main.py

### Alice FastAPI v1

This Python cell embeds Alice's older FastAPI project. It uses `requests.Request(..., json=...)`, which requires a newer Requests release than Bob's system copy.

In [ ]:
ALICE_V1 = Path('/home/alice/projects/fastapi-v1')
ALICE_V1_TMP = Path('/tmp/fastapi-v1-main.py')

alice_v1_code = '''# Alice FastAPI project 1
# FastAPI 0.68.2 + Pydantic 1 + newer Requests.
import sys
import fastapi
import requests
from fastapi import FastAPI

app = FastAPI(title='alice-fastapi-v1')

@app.get('/')
def root():
    request = requests.Request(
        'POST',
        'https://example.com',
        json={'project': 'alice-fastapi-v1'},
    )
    return {
        'project': 'alice-fastapi-v1',
        'fastapi': fastapi.__version__,
        'requests': requests.__version__,
        'request': request.method,
    }

if __name__ == '__main__':
    print('Alice FastAPI v1')
    print(f'  Python  : {sys.executable}')
    print(f'  fastapi : {fastapi.__version__}')
    print(f'  requests: {requests.__version__}')
    request = requests.Request(
        'POST',
        'https://example.com',
        json={'project': 'alice-fastapi-v1'},
    )
    print(f'  request : {request.method}')
'''

ALICE_V1_TMP.write_text(alice_v1_code, encoding='utf-8')

With Alice FastAPI v1 defined, this Bash block creates the project directory on the host and copies the generated source file into place.

In [ ]:
%%bash
set -euo pipefail
sudo mkdir -p /home/alice/projects/fastapi-v1
sudo cp /tmp/fastapi-v1-main.py /home/alice/projects/fastapi-v1/main.py
sudo chown -R alice:alice /home/alice/projects
ls -la /home/alice/projects/fastapi-v1
echo
echo 'main.py'
echo '-------'
cat /home/alice/projects/fastapi-v1/main.py

### Alice FastAPI v2

This Python cell embeds Alice's newer FastAPI project. It uses Pydantic v2's `model_dump()` API, which makes the FastAPI dependency conflict concrete when Alice shares one `--user` package set across both projects.

In [ ]:
ALICE_V2 = Path('/home/alice/projects/fastapi-v2')
ALICE_V2_TMP = Path('/tmp/fastapi-v2-main.py')

alice_v2_code = '''# Alice FastAPI project 2
# FastAPI 0.111.1 + Pydantic 2 + newer Requests.
import sys
import fastapi
import requests
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title='alice-fastapi-v2')

class Project(BaseModel):
    name: str
    version: str

@app.get('/')
def root():
    project = Project(name='alice-fastapi-v2', version=fastapi.__version__)
    return {
        'project': project.model_dump(),
        'fastapi': fastapi.__version__,
        'requests': requests.__version__,
    }

if __name__ == '__main__':
    print('Alice FastAPI v2')
    print(f'  Python  : {sys.executable}')
    print(f'  fastapi : {fastapi.__version__}')
    print(f'  requests: {requests.__version__}')
    project = Project(name='alice-fastapi-v2', version=fastapi.__version__)
    print(f'  model   : {project.model_dump()}')
    print('  API     : Pydantic v2 model_dump() supported')
'''

ALICE_V2_TMP.write_text(alice_v2_code, encoding='utf-8')

With Alice FastAPI v2 defined, this Bash block creates the project directory on the host and copies the generated source file into place.

In [ ]:
%%bash
set -euo pipefail
sudo mkdir -p /home/alice/projects/fastapi-v2
sudo cp /tmp/fastapi-v2-main.py /home/alice/projects/fastapi-v2/main.py
sudo chown -R alice:alice /home/alice/projects
ls -la /home/alice/projects/fastapi-v2
echo
echo 'main.py'
echo '-------'
cat /home/alice/projects/fastapi-v2/main.py

---

## Unveil Dependency Conflicts

### Shared System Interpreter Conflict

Bob and Alice both run the same system interpreter, so one shared `site-packages` directory has to satisfy two users at once.

Bob's legacy application starts from the baseline where `requests==2.0.0` is installed, but Alice needs a newer `requests==2.32.3`. That version mismatch makes the cross-user conflict visible as soon as the shared system package is changed.

In [ ]:
%%bash
set -euo pipefail
python3 -c "import requests, os; print('version :', requests.__version__); print('location:', os.path.dirname(requests.__file__))"
python3 /home/bob/projects/legacy-server/main.py

Alice shares that same interpreter, so her project starts with Bob's older dependency version and fails before she can use the newer Requests API it expects.

In [ ]:
%%bash
set -euo pipefail
sudo su - alice -c 'python3 /home/alice/projects/fastapi-v1/main.py' || true

Alice cannot repair the shared system interpreter herself. A direct attempt to replace the system package still targets a root-owned location, so only Bob can perform a machine-wide change.

That also explains the risk: if Bob upgraded the shared `requests` package for Alice, Bob's legacy project would immediately start running against the new version too. On a shared system interpreter, any package upgrade changes the dependency set for both users at once.

In [ ]:
%%bash
set -euo pipefail
sudo su - alice -c 'python3 -m pip install --break-system-packages --prefix=/usr/local --upgrade requests==2.32.3' || true

### Shared Python User Environment Conflicts

Now move to Alice's user site. This protects Bob from Alice's package installs, but Alice still has only one shared `--user` package set for all of her own projects. One person working on multiple projects can still hit dependency conflicts when all of those projects share the same user-level environment.

In [ ]:
%%bash
set -euo pipefail
sudo su - alice -c '
python3 -m pip install --user --upgrade requests==2.32.3
echo "user site: $(python3 -m site --user-site)"
python3 -c "import requests, os; print(\"version :\", requests.__version__); print(\"location:\", os.path.dirname(requests.__file__))"
'
echo
echo 'Bob still sees the system copy:'
python3 -c "import requests, os; print('version :', requests.__version__); print('location:', os.path.dirname(requests.__file__))"
python3 /home/bob/projects/legacy-server/main.py

> `pip install --user` isolates users from each other. Bob stays on the system copy, while Alice gets her own user-level `requests` installation. The next problem is that Alice's two projects still share the same `~/.local` package directory.

Start with Alice's older FastAPI dependency set. The old FastAPI line and Pydantic v1 support project v1, but that same shared user environment breaks project v2.

### FastAPI versions collide inside `--user`

In [ ]:
%%bash
sudo su - alice -c '
python3 -m pip install --user "fastapi==0.68.2" "pydantic<2"
echo "shared user fastapi: $(python3 -c "import fastapi; print(fastapi.__version__)")"
echo
echo "Run alice-fastapi-v1"
python3 /home/alice/projects/fastapi-v1/main.py
echo
echo "Run alice-fastapi-v2"
python3 /home/alice/projects/fastapi-v2/main.py
' || true

In [ ]:
%%bash
sudo su - alice -c '
python3 -m pip install --user --upgrade "fastapi==0.111.1" "pydantic>=2"
echo "shared user fastapi: $(python3 -c "import fastapi; print(fastapi.__version__)")"
echo
echo "Run alice-fastapi-v2"
python3 /home/alice/projects/fastapi-v2/main.py
echo
echo "Run alice-fastapi-v1"
python3 /home/alice/projects/fastapi-v1/main.py
' || true

> `--user` isolates users, **but not projects**. Alice cannot keep two incompatible FastAPI dependency sets in one shared `~/.local` installation.

---

## Resolving the Conflicts with Virtual Environments

Each project now gets its own `.venv`. The Bash cells below use explicit `.venv/bin/python` paths so the interpreter choice stays visible and no activation shell is needed.

In [ ]:
%%bash
set -euo pipefail
sudo su - bob -c '
python3 -m venv /home/bob/projects/legacy-server/.venv
/home/bob/projects/legacy-server/.venv/bin/python -m pip install --quiet --upgrade pip
/home/bob/projects/legacy-server/.venv/bin/python -m pip install --quiet "requests==2.0.0"
'

In [ ]:
%%bash
set -euo pipefail
sudo su - alice -c '
python3 -m venv /home/alice/projects/fastapi-v1/.venv
/home/alice/projects/fastapi-v1/.venv/bin/python -m pip install --quiet --upgrade pip
/home/alice/projects/fastapi-v1/.venv/bin/python -m pip install --quiet "fastapi==0.68.2" "pydantic<2" "requests==2.32.3"
python3 -m venv /home/alice/projects/fastapi-v2/.venv
/home/alice/projects/fastapi-v2/.venv/bin/python -m pip install --quiet --upgrade pip
/home/alice/projects/fastapi-v2/.venv/bin/python -m pip install --quiet "fastapi==0.111.1" "requests==2.32.3"
'

### The final state: three projects, three interpreters

Each application now runs with its own interpreter and its own package set. The output below should show different executable paths and independent dependency versions.

In [ ]:
%%bash
set -euo pipefail
echo 'Bob Legacy'
echo '----------'
sudo su - bob -c '/home/bob/projects/legacy-server/.venv/bin/python /home/bob/projects/legacy-server/main.py'
echo
echo 'Alice FastAPI V1'
echo '----------------'
sudo su - alice -c '/home/alice/projects/fastapi-v1/.venv/bin/python /home/alice/projects/fastapi-v1/main.py'
echo
echo 'Alice FastAPI V2'
echo '----------------'
sudo su - alice -c '/home/alice/projects/fastapi-v2/.venv/bin/python /home/alice/projects/fastapi-v2/main.py'